# 🚕 Doğrusal Regresyon — Chicago Taksi Ücreti Tahmini

ML Crash Course'un [programlama alıştırmasının](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/cc/exercises/linear_regression_taxi.ipynb) benim yeniden yazdığım hali.

**Kursun Colab'ından farkı:** Kursta model Keras ile kuruluyor. Burada aynı deneyleri **NumPy ile sıfırdan** yazdım, böylece önceki derslerde öğrendiğim gradyan inişi, öğrenme oranı ve grup boyutu kodda açıkça görünüyor. Kursun Keras modeli **RMSprop** optimizer'ı kullandığı için burada da varsayılan olarak RMSprop var; istenirse düz `sgd` de seçilebilir.

[![Colab'da aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cengizhankkaya/ml-crash-course-notlar/blob/main/kod/01-dogrusal-regresyon/taksi_ucreti.ipynb)

> ⚠️ Colab'ı açarken **tarayıcı çevirisini kapat**. Çeviri açıksa kod da çevriliyor (`pip install` → `pip yükleme`, `pandas` → `pandalar`, `0.001` → `0,001`) ve hücreler hata verir.

## Bölüm 1 — Kurulum

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:.2f}".format

In [ ]:
# Veri kümesi: Chicago taksi yolculukları, Mayıs 2022'den iki gün
URL = "https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv"
veri = pd.read_csv(os.environ.get("TAXI_CSV", URL))

# Kurstaki gibi yalnızca bu sütunları kullan
training_df = veri[["TRIP_MILES", "TRIP_SECONDS", "FARE", "COMPANY", "PAYMENT_TYPE", "TIP_RATE"]].copy()
print(f"{len(training_df):,} satır okundu")
training_df.head()

## Bölüm 2 — Veriyi tanıma

### İstatistikler
`describe()` sayısal sütunlar için ortalama, min, max gibi değerleri verir. `COMPANY` ve `PAYMENT_TYPE` **kategorik** (metin) olduğu için onlarda ortalama/max **NaN** görünür. Bu bir hata değil, anlamsız olduğu için hesaplanmıyor.

In [ ]:
training_df.describe(include="all")

In [ ]:
# Colab'daki 5 sorunun cevabı — kodla
print(f"1) En yüksek ücret        : ${training_df['FARE'].max():.2f}")
print(f"2) Ortalama mesafe        : {training_df['TRIP_MILES'].mean():.2f} mil")
print(f"3) Taksi şirketi sayısı   : {training_df['COMPANY'].nunique()}")
print(f"4) En sık ödeme türü      : {training_df['PAYMENT_TYPE'].mode()[0]}")
eksik = training_df.isnull().sum()
print(f"5) Eksik veri var mı?     : {'Evet' if eksik.any() else 'Hayır'}")

### Korelasyon matrisi
- **1.0** → mükemmel pozitif ilişki (biri artınca diğeri artar)
- **−1.0** → mükemmel negatif ilişki
- **0.0** → doğrusal ilişki yok

Mutlak değeri büyük olan özellik, etiketi tahmin etmede daha güçlüdür.

In [ ]:
korelasyon = training_df.corr(numeric_only=True)
display(korelasyon)

fare_kor = korelasyon["FARE"].drop("FARE").sort_values(key=abs, ascending=False)
print("FARE ile korelasyon (güçlüden zayıfa):")
print(fare_kor.to_string())
print(f"\nEn güçlü: {fare_kor.index[0]}   En zayıf: {fare_kor.index[-1]}")

### Çift grafiği (pair plot)
Her sayısal özelliğin diğerleriyle ilişkisini tek tabloda gösterir.

In [ ]:
pd.plotting.scatter_matrix(training_df[["FARE", "TRIP_MILES", "TRIP_SECONDS"]],
                           figsize=(8, 8), alpha=0.2, s=5)
plt.show()

## Bölüm 3 — Modeli eğitme

### Eğitim fonksiyonu (sıfırdan)
- `y' = b + w₁x₁ + w₂x₂ + ...`
- Kayıp: **MSE**, izlenen metrik: **RMSE** (etiketle aynı birimde, yani **dolar**)
- Hiperparametreler: `learning_rate`, `epochs`, `batch_size`
- Her dönemde veri karıştırılır ve mini gruplara bölünür (mini grup SGD)

In [ ]:
def egit(df, ozellikler, etiket="FARE", learning_rate=0.001, epochs=20, batch_size=50,
         optimizer="rmsprop", tohum=42):
    X = df[ozellikler].to_numpy(dtype=float)
    y = df[etiket].to_numpy(dtype=float)
    N, d = X.shape
    w, b = np.zeros(d), 0.0
    cache_w, cache_b = np.zeros(d), 0.0          # RMSprop için
    rng = np.random.default_rng(tohum)
    rmse_gecmisi = []

    with np.errstate(over="ignore", invalid="ignore"):
        for _ in range(epochs):
            sira = rng.permutation(N)
            for i in range(0, N, batch_size):
                j = sira[i:i + batch_size]
                hata = X[j] @ w + b - y[j]                 # y' - y
                gw = 2 * X[j].T @ hata / len(j)            # ∂MSE/∂w
                gb = 2 * hata.mean()                       # ∂MSE/∂b
                if optimizer == "rmsprop":                 # adımı gradyanın büyüklüğüne göre ölçekler
                    cache_w = 0.9 * cache_w + 0.1 * gw ** 2
                    cache_b = 0.9 * cache_b + 0.1 * gb ** 2
                    w -= learning_rate * gw / (np.sqrt(cache_w) + 1e-7)
                    b -= learning_rate * gb / (np.sqrt(cache_b) + 1e-7)
                else:                                      # düz SGD
                    w -= learning_rate * gw
                    b -= learning_rate * gb
            rmse_gecmisi.append(np.sqrt(np.mean((X @ w + b - y) ** 2)))

    ad = f"{'+'.join(ozellikler)} | lr={learning_rate}, grup={batch_size}"
    return {"ad": ad, "w": w, "b": b, "rmse": np.array(rmse_gecmisi), "ozellikler": ozellikler}


def ozet(deney):
    terimler = " + ".join(f"{w:.3f}·{o}" for w, o in zip(deney["w"], deney["ozellikler"]))
    print(f"[{deney['ad']}]")
    print(f"  Model: FARE' = {deney['b']:.3f} + {terimler}")
    print(f"  Son RMSE: ${deney['rmse'][-1]:.2f}")


def rmse_ciz(*deneyler):
    plt.figure(figsize=(8, 4))
    for d in deneyler:
        plt.plot(range(1, len(d["rmse"]) + 1), d["rmse"], marker="o", ms=3, label=d["ad"])
    plt.xlabel("Dönem (epoch)"); plt.ylabel("RMSE ($)"); plt.yscale("log")
    plt.grid(alpha=.3); plt.legend(); plt.show()

### Deney 1 — Tek özellik: `TRIP_MILES`
Korelasyon matrisinde ücretle en güçlü ilişkili özellik mesafeydi; ilk denemeyi onunla yapıyoruz.

`learning_rate=0.001`, `epochs=20`, `batch_size=50`

In [ ]:
deney_1 = egit(training_df, ["TRIP_MILES"], learning_rate=0.001, epochs=20, batch_size=50)
ozet(deney_1)
rmse_ciz(deney_1)

ornek = training_df.sample(500, random_state=1)
xs = np.linspace(0, ornek["TRIP_MILES"].max(), 50)
plt.figure(figsize=(8, 4))
plt.scatter(ornek["TRIP_MILES"], ornek["FARE"], s=6, alpha=.4, label="Veri (500 örnek)")
plt.plot(xs, deney_1["b"] + deney_1["w"][0] * xs, color="red", label="Model")
plt.xlabel("TRIP_MILES"); plt.ylabel("FARE ($)"); plt.legend(); plt.grid(alpha=.3); plt.show()

### Hiperparametre deneyleri
Kurstaki gibi her seferinde **tek** bir hiperparametreyi değiştiriyoruz:

| Deney | Öğrenme oranı | Grup boyutu |
|---|---|---|
| Temel (Deney 1) | 0.001 | 50 |
| A — oranı artır | **1.0** | 50 |
| B — oranı düşür | **0.0001** | 50 |
| C — grubu büyüt | 0.001 | **500** |

In [ ]:
deney_A = egit(training_df, ["TRIP_MILES"], learning_rate=1.0,    batch_size=50)
deney_B = egit(training_df, ["TRIP_MILES"], learning_rate=0.0001, batch_size=50)
deney_C = egit(training_df, ["TRIP_MILES"], learning_rate=0.001,  batch_size=500)
for d in (deney_1, deney_A, deney_B, deney_C):
    ozet(d)
rmse_ciz(deney_1, deney_A, deney_B, deney_C)

In [ ]:
# Karşılaştırma: aynı oranla (0.001) DÜZ SGD ne yapar?
deney_sgd = egit(training_df, ["TRIP_MILES"], learning_rate=0.001, optimizer="sgd")
ozet(deney_sgd)

### Deney 3 — İki özellik: `TRIP_MILES` + `TRIP_MINUTES`
Veride dakika yok ama saniyeden kolayca türetilir. Bu, **özellik mühendisliğinin** (feature engineering) ilk örneği.

In [ ]:
training_df["TRIP_MINUTES"] = training_df["TRIP_SECONDS"] / 60

deney_3 = egit(training_df, ["TRIP_MILES", "TRIP_MINUTES"], learning_rate=0.001, epochs=20, batch_size=50)
ozet(deney_3)

In [ ]:
# Soru: TRIP_MINUTES yerine TRIP_SECONDS kullansak fark eder mi?
deney_3s = egit(training_df, ["TRIP_MILES", "TRIP_SECONDS"], learning_rate=0.001, epochs=20, batch_size=50)
ozet(deney_3s)
print(f"\nSaniye ağırlığı × 60 = {deney_3s['w'][1] * 60:.3f}  (dakika ağırlığı: {deney_3['w'][1]:.3f})")

In [ ]:
# Deneyleri karşılaştır
rmse_ciz(deney_1, deney_3, deney_3s)

## Bölüm 4 — Tahmin yapma
Normalde tahmin, eğitimde **kullanılmamış** verilerle yapılmalı. Burada kurstaki gibi eğitim verisinden rastgele örnekler kullanıyoruz; doğru yöntem ileriki derslerde gelecek.

In [ ]:
def tahmin_et(deney, df):
    return df[deney["ozellikler"]].to_numpy(dtype=float) @ deney["w"] + deney["b"]

ornek = training_df.sample(10, random_state=7)
sonuc = pd.DataFrame({
    "MİL": ornek["TRIP_MILES"].values,
    "DAKİKA": ornek["TRIP_MINUTES"].values,
    "GERÇEK ÜCRET": ornek["FARE"].values,
    "TAHMİN": tahmin_et(deney_3, ornek),
})
sonuc["FARK ($)"] = (sonuc["TAHMİN"] - sonuc["GERÇEK ÜCRET"]).abs()
display(sonuc)
print(f"Ortalama mutlak fark (MAE): ${sonuc['FARK ($)'].mean():.2f}")

## 📝 Sonuçlarım
Colab'da çalıştırdıktan sonra buraya kendi gözlemlerimi yazacağım.

- Deney 1 kaç dönemde yakınsadı?
- Öğrenme oranı 1.0 / 0.0001 ne yaptı?
- Grup boyutu 500 neyi değiştirdi?
- İki özellik tek özellikten iyi mi?